In [1]:
import numpy as np
import tensorflow as tf
import random
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout,Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers

In [2]:
np.random.seed(10)
tf.random.set_seed(10)
random.seed(10)

In [3]:
data = pd.read_csv(r'E:\Dataset\credit_scoring_sample.csv').fillna(0)
print(data.head())

data_majority = data[data.SeriousDlqin2yrs == 0]
data_minority = data[data.SeriousDlqin2yrs == 1]

data_majority_downsampled = resample(data_majority, replace=False, n_samples=len(data_minority), random_state=10) 
data_balanced = pd.concat([data_majority_downsampled, data_minority])

y = data_balanced['SeriousDlqin2yrs'].values
X = data_balanced.drop('SeriousDlqin2yrs', axis=1).values

   SeriousDlqin2yrs  age  NumberOfTime30-59DaysPastDueNotWorse    DebtRatio  \
0                 0   64                                     0     0.249908   
1                 0   58                                     0  3870.000000   
2                 0   41                                     0     0.456127   
3                 0   43                                     0     0.000190   
4                 1   49                                     0     0.271820   

   NumberOfTimes90DaysLate  NumberOfTime60-89DaysPastDueNotWorse  \
0                        0                                     0   
1                        0                                     0   
2                        0                                     0   
3                        0                                     0   
4                        0                                     0   

   MonthlyIncome  NumberOfDependents  
0         8158.0                 0.0  
1            0.0                 0.0  

In [4]:
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=10)

In [5]:
baseline_model = Sequential()
baseline_model.add(Input(shape=(7,)))
baseline_model.add(Dense(8, activation='relu')) 
baseline_model.add(Dense(1, activation='sigmoid')) 

baseline_model.compile(
    optimizer='adam',
    loss='binary_crossentropy', 
    metrics=['accuracy']
)
baseline_model.fit(X_train, y_train, epochs=40, batch_size=10, verbose=1)

loss1, acc_before = baseline_model.evaluate(X_test, y_test, verbose=1)

Epoch 1/40
1404/1404 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.6048 - loss: 0.6642
Epoch 2/40
1404/1404 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.6491 - loss: 0.6306
Epoch 3/40
1404/1404 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.7010 - loss: 0.5901
Epoch 4/40
1404/1404 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.7248 - loss: 0.5624
Epoch 5/40
1404/1404 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.7375 - loss: 0.5468
Epoch 6/40
1404/1404 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.7464 - loss: 0.5364
Epoch 7/40
1404/1404 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.7477 - loss: 0.5290
Epoch 8/40
1404/1404 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.7486 - loss: 0.5252
Epoch 9/40
1404/1404 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.7515 - loss: 0.5226
Epoch 10/40
1404/1404 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.7523 - loss: 0.5209
Epoch 11/40
1404/1404 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.7527 - loss: 0.5195
Epoch 12/40
1404/1404 ━━━━━━━━

In [6]:
reg_model = Sequential()
reg_model.add(Input(shape=(7,)))
reg_model.add(Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
reg_model.add(Dropout(0.3))
reg_model.add(Dense(16, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
reg_model.add(Dropout(0.3))
reg_model.add(Dense(1, activation='sigmoid')) 

reg_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

reg_model.fit(X_train, y_train, epochs=200, batch_size=10, validation_split=0.2, callbacks=[early_stop], verbose=1)

loss2, acc_after = reg_model.evaluate(X_test, y_test, verbose=1)

Epoch 1/200
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5898 - loss: 0.6978 - val_accuracy: 0.6738 - val_loss: 0.6290
Epoch 2/200
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6773 - loss: 0.6426 - val_accuracy: 0.7201 - val_loss: 0.5836
Epoch 3/200
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6945 - loss: 0.6207 - val_accuracy: 0.7308 - val_loss: 0.5688
Epoch 4/200
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.7147 - loss: 0.5965 - val_accuracy: 0.7397 - val_loss: 0.5602
Epoch 5/200
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7203 - loss: 0.5858 - val_accuracy: 0.7415 - val_loss: 0.5560
Epoch 6/200
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.7278 - loss: 0.5754 - val_accuracy: 0.7475 - val_loss: 0.5493
Epoch 7/200
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.7356 - loss: 0.5738 - val_accuracy: 0.7457 - val_loss: 0.5478
Epoch 8/200
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7347 - loss: 0

In [7]:
print("\n===== Accuracy Comparison =====")
print("Accuracy BEFORE Regularization : {:.2f}%".format(acc_before * 100))
print("Accuracy AFTER Regularization : {:.2f}%".format(acc_after * 100))


===== Accuracy Comparison =====
Accuracy BEFORE Regularization : 75.63%
Accuracy AFTER Regularization : 75.28%


In [8]:
y_pred = (reg_model.predict(X_test) > 0.5).astype("int32")

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Detailed Report ---")
print(classification_report(y_test, y_pred))

188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 658us/step
--- Confusion Matrix ---
[[2563  444]
 [1043 1966]]

--- Detailed Report ---
              precision    recall  f1-score   support

           0       0.71      0.85      0.78      3007
           1       0.82      0.65      0.73      3009

    accuracy                           0.75      6016
   macro avg       0.76      0.75      0.75      6016
weighted avg       0.76      0.75      0.75      6016

